chargement des librairies presentes dans le fichier src/imports.py

In [ ]:
import sys
sys.path.append('..')
from src.imports import *

importation des modules pour le prétraitement des données : séparation et mise à l’échelle

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

chargement du dataset

In [42]:
df = pd.read_csv("data/raw/cybersecurity_intrusion_data.csv")

Supression de la colonne User_id

In [43]:
df = df.drop(columns=["session_id"])

Suppression des valeurs manquantes 

In [45]:
df = df.dropna()
df.shape

(7571, 10)

One hot encoding

In [46]:
df = pd.get_dummies(df, columns=['protocol_type'])
df = pd.get_dummies(df, columns=['encryption_used'])
df = pd.get_dummies(df, columns=['browser_type'])

In [47]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

separation des donnees d'entrainements et de test

In [48]:
X = df.drop('attack_detected', axis = 1)
y = df['attack_detected']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [49]:
#log transformation pour réduire l'impact des valeurs extrêmes
X_train['session_duration'] = np.log1p(X_train['session_duration'])
X_test['session_duration'] = np.log1p(X_test['session_duration'])

#normalisation des données 
cols_to_scale = ['network_packet_size', 'session_duration', 'login_attempts']
scaler = MinMaxScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

Apercu de nos nouvelles données prétraitées

In [50]:
X_train.head()

,network_packet_size,login_attempts,session_duration,ip_reputation_score,failed_logins,unusual_time_access,protocol_type_ICMP,protocol_type_TCP,protocol_type_UDP,encryption_used_AES,encryption_used_DES,browser_type_Chrome,browser_type_Edge,browser_type_Firefox,browser_type_Safari,browser_type_Unknown
3159,0.407862,0.166667,0.664269,0.357243,0,0,0,1,0,1,0,1,0,0,0,0
3873,0.114660,0.333333,0.580400,0.385465,1,0,0,0,1,0,1,1,0,0,0,0
3034,0.366093,0.250000,0.796863,0.480513,0,0,0,0,1,1,0,0,0,0,0,1
504,0.249795,0.250000,0.508873,0.344468,2,0,0,1,0,0,1,1,0,0,0,0
2681,0.307125,0.500000,0.629873,0.392859,0,0,0,0,1,0,1,1,0,0,0,0


Enregistrement des donnees 

In [51]:
X_train.to_csv("data/Processed/X_train.csv", index= False)
X_test.to_csv("data/Processed/X_test.csv", index= False)
y_train.to_csv("data/Processed/y_train.csv", index= False)
y_test.to_csv("data/Processed/y_test.csv", index= False)